# 06 - Analise Estatistica da Segmentacao Binarizada

Constroi a base analitica da segmentacao binarizada a partir do SQLite, calcula os resumos estatisticos em memoria e salva tabelas CSV para a etapa visual do notebook 07.


In [ ]:
import sys
from pathlib import Path

import pandas as pd

root_dir = Path.cwd()
if not (root_dir / 'src').exists() and (root_dir.parent / 'src').exists():
    root_dir = root_dir.parent

if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.analysis.bootstrap import build_bootstrap_confidence_intervals
from src.analysis.descriptive_stats import MetricConfig, build_descriptive_stats
from src.analysis.stability import build_execution_stability
from src.analysis.statistical_tests import (
    GLOBAL_SCOPE,
    build_model_comparison_tests,
    build_model_tag_interactions,
    build_tag_impact_tests,
)
from src.repositories import ImagemRepository


## Carregamento da base analitica em memoria

Le a avaliacao persistida no SQLite e monta uma base por `imagem + modelo + execucao + estrategia_binarizacao`, focada nas metricas `iou`, `precision` e `recall`.


In [ ]:
BINARIZED_METRIC_CONFIGS = (
    MetricConfig(metric_name='iou', higher_is_better=True),
    MetricConfig(metric_name='precision', higher_is_better=True),
    MetricConfig(metric_name='recall', higher_is_better=True),
)

TAG_COLUMNS = (
    'ok',
    'multi_bufalos',
    'cortado',
    'angulo_extremo',
    'baixo_contraste',
    'ocluido',
)


def _build_segmentacao_binarizada_base(imagens):
    registros = []

    for imagem in imagens:
        tags = sorted(dict.fromkeys(imagem.nomes_tags))
        tags_sem_ok = [tag for tag in tags if tag != 'ok']
        num_tags_problema = len(tags_sem_ok)

        if not tags:
            grupo_dificuldade = 'nao_revisada'
        elif 'ok' in tags and num_tags_problema == 0:
            grupo_dificuldade = 'ok'
        elif num_tags_problema <= 1:
            grupo_dificuldade = '1_problema'
        else:
            grupo_dificuldade = '2_ou_mais_problemas'

        for segmentacao_bruta in imagem.segmentacoes_brutas:
            for segmentacao_binarizada in segmentacao_bruta.segmentacoes_binarizadas:
                if (
                    segmentacao_binarizada.iou is None
                    or segmentacao_binarizada.precision is None
                    or segmentacao_binarizada.recall is None
                ):
                    continue

                registro = {
                    'nome_arquivo': imagem.nome_arquivo,
                    'fazenda': imagem.fazenda,
                    'peso': imagem.peso,
                    'modelo': segmentacao_bruta.nome_modelo,
                    'execucao': segmentacao_bruta.execucao,
                    'estrategia_binarizacao': segmentacao_binarizada.estrategia_binarizacao,
                    'iou': float(segmentacao_binarizada.iou),
                    'precision': float(segmentacao_binarizada.precision),
                    'recall': float(segmentacao_binarizada.recall),
                    'area': float(segmentacao_binarizada.area),
                    'perimetro': float(segmentacao_binarizada.perimetro),
                    'tags': ','.join(tags),
                    'tags_sem_ok': ','.join(tags_sem_ok),
                    'num_tags_problema': num_tags_problema,
                    'tem_tag_problema': num_tags_problema > 0,
                    'grupo_dificuldade': grupo_dificuldade,
                }

                for tag_name in TAG_COLUMNS:
                    registro[f'tag_{tag_name}'] = tag_name in tags

                registros.append(registro)

    return pd.DataFrame(registros)


def _build_tag_analysis_base(df_base):
    tag_columns = [column for column in df_base.columns if column.startswith('tag_')]
    if not tag_columns:
        raise ValueError('DataFrame base nao contem colunas de tag.')

    return df_base.melt(
        id_vars=[column for column in df_base.columns if column not in tag_columns],
        value_vars=tag_columns,
        var_name='tag_name',
        value_name='tag_value',
    )


def _build_execution_stability_modelo_estrategia(df_base, metric_configs):
    rows = []

    for metric_config in metric_configs:
        per_execution = (
            df_base.groupby(['modelo', 'estrategia_binarizacao', 'execucao'], dropna=False)[metric_config.metric_name]
            .mean()
            .reset_index(name='execution_mean')
        )

        for (model_name, strategy_name), group_df in per_execution.groupby(
            ['modelo', 'estrategia_binarizacao'],
            dropna=False,
        ):
            execution_means = group_df['execution_mean'].astype(float)
            mean_execucoes = float(execution_means.mean())
            std_execucoes = float(execution_means.std(ddof=1)) if len(execution_means) > 1 else 0.0
            cv_execucoes = std_execucoes / abs(mean_execucoes) if mean_execucoes != 0.0 else 0.0
            amplitude_execucoes = float(execution_means.max() - execution_means.min())

            ascending = not metric_config.higher_is_better
            ordered = group_df.sort_values('execution_mean', ascending=ascending)

            rows.append(
                {
                    'modelo': str(model_name),
                    'estrategia_binarizacao': str(strategy_name),
                    'metric_name': metric_config.metric_name,
                    'count_execucoes': int(group_df['execucao'].nunique()),
                    'mean_execucoes': mean_execucoes,
                    'std_execucoes': std_execucoes,
                    'cv_execucoes': cv_execucoes,
                    'amplitude_execucoes': amplitude_execucoes,
                    'melhor_execucao': int(ordered.iloc[0]['execucao']),
                    'pior_execucao': int(ordered.iloc[-1]['execucao']),
                    'higher_is_better': metric_config.higher_is_better,
                }
            )

    return pd.DataFrame(rows)


def _build_strategy_comparison_tests(df_base, metric_configs):
    strategy_base = df_base.copy()
    strategy_base['modelo_origem'] = strategy_base['modelo']
    strategy_base['modelo'] = strategy_base['estrategia_binarizacao']

    frames = []

    global_tests = build_model_comparison_tests(
        df_base=strategy_base.drop(columns=['modelo_origem']),
        metric_configs=metric_configs,
    )
    if not global_tests.empty:
        global_tests['modelo_origem'] = GLOBAL_SCOPE
        frames.append(global_tests)

    for model_name, model_df in strategy_base.groupby('modelo_origem', dropna=False):
        model_tests = build_model_comparison_tests(
            df_base=model_df.drop(columns=['modelo_origem']),
            metric_configs=metric_configs,
        )
        if model_tests.empty:
            continue

        model_tests['modelo_origem'] = str(model_name)
        frames.append(model_tests)

    if not frames:
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=True)


def _build_tag_impact_tests_by_strategy(df_base, metric_configs):
    strategy_base = df_base.copy()
    strategy_base['modelo'] = strategy_base['estrategia_binarizacao']
    result = build_tag_impact_tests(
        df_base=strategy_base,
        metric_configs=metric_configs,
    )
    if result.empty:
        return result

    return result.rename(columns={'nome_modelo': 'estrategia_binarizacao'})


def _build_tag_interactions_by_strategy(df_base, metric_configs):
    strategy_base = df_base.copy()
    strategy_base['modelo'] = strategy_base['estrategia_binarizacao']
    result = build_model_tag_interactions(
        df_base=strategy_base,
        metric_configs=metric_configs,
    )
    if result.empty:
        return result

    return result.rename(columns={'nome_modelo': 'estrategia_binarizacao'})


imagem_repository = ImagemRepository()
df_base = _build_segmentacao_binarizada_base(imagem_repository.list())

print(f'Registros na base analitica: {len(df_base)}')
print(f'Imagens: {df_base["nome_arquivo"].nunique()}')
print(f'Modelos: {df_base["modelo"].nunique()}')
print(f'Execucoes: {df_base["execucao"].nunique()}')
print(f'Estrategias de binarizacao: {df_base["estrategia_binarizacao"].nunique()}')
print(f'Tags binarias disponiveis: {len([column for column in df_base.columns if column.startswith("tag_")])}')

df_base.head()


## Calculo e persistencia em CSV

Gera as tabelas analiticas da segmentacao binarizada e salva os resultados em `generated/analysis/segmentacao_binarizada`, preparando a leitura do notebook 07.


In [ ]:
output_dir = root_dir / 'generated' / 'analysis' / 'segmentacao_binarizada'
output_dir.mkdir(parents=True, exist_ok=True)

df_tag_base = _build_tag_analysis_base(df_base)

df_resumo_estrategia = build_descriptive_stats(
    df_base=df_base,
    metric_configs=BINARIZED_METRIC_CONFIGS,
    group_by=['estrategia_binarizacao'],
)
df_resumo_modelo_estrategia = build_descriptive_stats(
    df_base=df_base,
    metric_configs=BINARIZED_METRIC_CONFIGS,
    group_by=['modelo', 'estrategia_binarizacao'],
)
df_resumo_execucao = build_descriptive_stats(
    df_base=df_base,
    metric_configs=BINARIZED_METRIC_CONFIGS,
    group_by=['modelo', 'estrategia_binarizacao', 'execucao'],
)
df_resumo_tag = build_descriptive_stats(
    df_base=df_tag_base,
    metric_configs=BINARIZED_METRIC_CONFIGS,
    group_by=['modelo', 'estrategia_binarizacao', 'tag_name', 'tag_value'],
)
df_estabilidade_modelo = build_execution_stability(
    df_base=df_base,
    metric_configs=BINARIZED_METRIC_CONFIGS,
)
df_estabilidade_modelo_estrategia = _build_execution_stability_modelo_estrategia(
    df_base=df_base,
    metric_configs=BINARIZED_METRIC_CONFIGS,
)
df_intervalo_confianca_modelo_estrategia = build_bootstrap_confidence_intervals(
    df_base=df_base,
    metric_configs=BINARIZED_METRIC_CONFIGS,
    group_by=['modelo', 'estrategia_binarizacao'],
)
df_testes_estrategia = _build_strategy_comparison_tests(
    df_base=df_base,
    metric_configs=BINARIZED_METRIC_CONFIGS,
)
df_testes_tag_estrategia = _build_tag_impact_tests_by_strategy(
    df_base=df_base,
    metric_configs=BINARIZED_METRIC_CONFIGS,
)
df_interacoes_tag_estrategia = _build_tag_interactions_by_strategy(
    df_base=df_base,
    metric_configs=BINARIZED_METRIC_CONFIGS,
)

artefatos = {
    'base_analitica.csv': df_base,
    'resumo_estrategia.csv': df_resumo_estrategia,
    'resumo_modelo_estrategia.csv': df_resumo_modelo_estrategia,
    'resumo_execucao.csv': df_resumo_execucao,
    'resumo_tag.csv': df_resumo_tag,
    'estabilidade_modelo.csv': df_estabilidade_modelo,
    'estabilidade_modelo_estrategia.csv': df_estabilidade_modelo_estrategia,
    'intervalo_confianca_modelo_estrategia.csv': df_intervalo_confianca_modelo_estrategia,
    'testes_estrategia.csv': df_testes_estrategia,
    'testes_tag_estrategia.csv': df_testes_tag_estrategia,
    'interacao_tag_estrategia.csv': df_interacoes_tag_estrategia,
}

for file_name, dataframe in artefatos.items():
    dataframe.to_csv(output_dir / file_name, index=False)

print(f'Arquivos salvos em: {output_dir}')
for file_name, dataframe in artefatos.items():
    print(f'- {file_name}: {len(dataframe)} linhas')


## Leitura inicial dos resultados

Mostra uma leitura compacta das tabelas calculadas para validar o material que sera explorado graficamente no notebook 07.


In [ ]:
pd.pivot_table(
    df_resumo_estrategia,
    index='estrategia_binarizacao',
    columns='metric_name',
    values=['mean', 'median'],
)


In [ ]:
df_resumo_modelo_estrategia.sort_values(['metric_name', 'modelo', 'mean'], ascending=[True, True, False]).head(24)


In [ ]:
df_estabilidade_modelo_estrategia.sort_values(['metric_name', 'cv_execucoes', 'amplitude_execucoes'])


In [ ]:
df_intervalo_confianca_modelo_estrategia.sort_values(
    ['metric_name', 'statistic_name', 'modelo', 'estrategia_binarizacao']
).head(24)


In [ ]:
if df_testes_estrategia.empty:
    print('Nao ha estrategias suficientes no banco atual para executar comparacoes estatisticas.')
else:
    df_testes_estrategia.sort_values(
        ['metric_name', 'modelo_origem', 'comparison_scope', 'p_value_adjusted']
    ).head(30)


In [ ]:
df_testes_tag_estrategia.sort_values(
    ['metric_name', 'estrategia_binarizacao', 'tag_name', 'comparison_scope', 'p_value_adjusted']
).head(30)


In [ ]:
df_interacoes_tag_estrategia.sort_values(
    ['metric_name', 'estrategia_binarizacao', 'adjusted_delta_mean']
).head(30)
